# 00 — Definisanje problema

## Telco Customer Churn: Predviđanje odliva korisnika

Ova sveska predstavlja **prvi korak** u analizi i uvodi problem koji rešavamo.
Pre bilo kakve obrade podataka, potrebno je jasno definisati *šta* predviđamo,
*zašto* je to važno i *kako* ćemo meriti uspeh.

Sveska ne sadrži obradu podataka — ona je konceptualni uvod u projekat.

## 1. Kontekst i motivacija

Telekomunikacione kompanije posluju na izuzetno konkurentnom tržištu na kojem
korisnici lako prelaze od jednog provajdera ka drugom. Pojava kada korisnik
napusti kompaniju naziva se **odliv korisnika** (engl. *customer churn*).

Odliv je značajan poslovni problem iz jednostavnog razloga: **zadržavanje
postojećeg korisnika je znatno jeftinije nego pridobijanje novog.** Umesto da
kompanija troši sredstva na privlačenje novih korisnika, isplativije je
prepoznati postojeće korisnike koji su u riziku od odlaska i zadržati ih
ciljanim merama (popust, bolja ponuda, kontakt korisničke podrške).

Ključni preduslov za to je **sposobnost da se unapred predvidi ko će otići.**
Upravo to je cilj ovog projekta: na osnovu podataka o korisniku predvideti
da li će on napustiti kompaniju, kako bi kompanija mogla da deluje proaktivno.

## 2. Cilj rada

Cilj je izgraditi model koji, na osnovu poznatih podataka o korisniku,
predviđa vrednost ciljne promenljive **`Churn`** — da li je korisnik napustio
kompaniju (`Yes`) ili je ostao (`No`).

Sa stanovišta mašinskog učenja, ovo je problem:

- **Nadgledanog učenja (supervised learning)** — jer za svakog korisnika u
  skupu podataka već znamo tačan odgovor (da li je otišao ili ne), pa model
  uči iz označenih primera.
- **Binarne klasifikacije** — jer ciljna promenljiva ima tačno dve moguće
  vrednosti (`Yes` / `No`).

Model dakle ne predviđa broj (to bi bila regresija), već **kategoriju** —
svrstava korisnika u jednu od dve klase.

## 3. Opis skupa podataka

Skup podataka **Telco Customer Churn** sadrži informacije o **7.043 korisnika**
telekomunikacione kompanije, opisane kroz **21 kolonu**. Za svakog korisnika
poznato je ko je, koje usluge koristi, kakav ugovor ima, kako i koliko plaća,
i — najvažnije — da li je napustio kompaniju.

Kolone se mogu konceptualno podeliti u **četiri grupe**:

**A. Demografske informacije** — govore o samom korisniku:
`gender`, `SeniorCitizen`, `Partner`, `Dependents`

**B. Usluge koje korisnik koristi** — koje telekomunikacione usluge ima:
`PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`,
`OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`

**C. Informacije o nalogu i plaćanju** — odnos korisnika sa kompanijom:
`tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`,
`MonthlyCharges`, `TotalCharges`

**D. Ciljna promenljiva:**
`Churn` — da li je korisnik napustio kompaniju u prethodnom mesecu.

### Detaljan opis kolona

| Kolona | Opis | Tip |
|---|---|---|
| `customerID` | Jedinstveni identifikator korisnika. Ne utiče na churn — izbacićemo ga iz modela. | Identifikator |
| `gender` | Pol korisnika: `Female`, `Male`. | Kategorijska (nominalna) |
| `SeniorCitizen` | Da li je korisnik stariji građanin: `0` = ne, `1` = da. Iako je zapisana kao broj, predstavlja kategoriju. | Kategorijska (binarna) |
| `Partner` | Da li korisnik ima partnera: `Yes`, `No`. | Kategorijska (binarna) |
| `Dependents` | Da li korisnik izdržava druge osobe (npr. decu): `Yes`, `No`. | Kategorijska (binarna) |
| `tenure` | Broj meseci koliko je korisnik kod kompanije. | Numerička (diskretna) |
| `PhoneService` | Da li korisnik ima telefonsku uslugu: `Yes`, `No`. | Kategorijska (binarna) |
| `MultipleLines` | Da li ima više telefonskih linija: `Yes`, `No`, `No phone service`. | Kategorijska (nominalna) |
| `InternetService` | Tip internet usluge: `DSL`, `Fiber optic`, `No`. | Kategorijska (nominalna) |
| `OnlineSecurity` | Dodatna online zaštita: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `OnlineBackup` | Online backup: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `DeviceProtection` | Zaštita uređaja: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `TechSupport` | Tehnička podrška: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `StreamingTV` | Streaming televizije: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `StreamingMovies` | Streaming filmova: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `Contract` | Tip ugovora: `Month-to-month`, `One year`, `Two year`. | Kategorijska (ordinalna) |
| `PaperlessBilling` | Elektronsko plaćanje bez papira: `Yes`, `No`. | Kategorijska (binarna) |
| `PaymentMethod` | Način plaćanja: `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)`. | Kategorijska (nominalna) |
| `MonthlyCharges` | Mesečni iznos naplate. | Numerička (kontinualna) |
| `TotalCharges` | Ukupan iznos naplaćen do sada. | Numerička (kontinualna) |
| `Churn` | **Ciljna promenljiva** — da li je korisnik napustio kompaniju: `Yes`, `No`. | Kategorijska (binarna) |

## 4. Izazovi u podacima

Već na osnovu inicijalnog upoznavanja sa skupom podataka, možemo predvideti
nekoliko izazova koje treba rešiti tokom analize:

**1. Skrivene nedostajuće vrednosti u koloni `TotalCharges`.**
Kolona `TotalCharges` je zapisana kao tekst iako predstavlja broj. Razlog je
to što 11 korisnika ima prazan razmak umesto vrednosti. Ove vrednosti se ne
prepoznaju standardnom proverom nedostajućih vrednosti (`isnull()`), već
zahtevaju eksplicitnu detekciju i konverziju u numerički tip.

**2. Nebalansirane klase ciljne promenljive.**
Korisnici koji su ostali (`Churn = No`) čine oko 73% skupa, dok oni koji su
otišli (`Churn = Yes`) čine oko 27%. Ovakav nebalans znači da obična tačnost
(*accuracy*) nije pouzdana metrika i da će modeliranje zahtevati posebne
strategije (npr. balansiranje klasa, izbor prikladnih metrika).

**3. Kategorije `"No internet service"` i `"No phone service"`.**
Više kolona (npr. `OnlineSecurity`, `StreamingTV`) ima treću vrednost poput
`No internet service`. Važno je uočiti da ova vrednost **nije isto što i** `No`:
- `No` znači da korisnik ima internet, ali ne koristi tu dodatnu uslugu.
- `No internet service` znači da korisnik uopšte nema internet.

Ova razlika će zahtevati pažljivu odluku tokom pripreme podataka.

## 5. Merilo uspeha

Pošto su klase nebalansirane (~73% naspram ~27%), **tačnost (accuracy) nije
dovoljna metrika.** Model koji bi uvek predviđao `No` postigao bi ~73% tačnosti,
a bio bi potpuno beskoristan — ne bi prepoznao nijednog korisnika u riziku od
odlaska.

Zbog toga ćemo kvalitet modela ocenjivati sledećim metrikama:

- **Precision (preciznost)** — od svih korisnika koje je model označio kao
  „otići će", koliko je zaista otišlo. Meri koliko su alarmi tačni.
- **Recall (odziv)** — od svih korisnika koji su zaista otišli, koliko je
  model uspeo da uhvati. Meri koliko odlazaka propuštamo.
- **F1-score** — harmonijska sredina precision-a i recall-a; jedinstvena mera
  koja balansira ta dva.
- **AUC-ROC** — sposobnost modela da razlikuje dve klase, nezavisno od izbora
  praga odlučivanja.

Poseban značaj ima **recall za klasu `Yes`**: sa poslovne strane, propustiti
korisnika koji će otići (a ne reagovati) obično je skuplje nego lažno označiti
lojalnog korisnika. Tokom evaluacije vodićemo računa o ovom balansu.

Sve metrike biće računate **isključivo na test skupu**, koji neće biti korišćen
ni u jednoj fazi pripreme podataka ili treniranja modela.

## 6. Prvi pogled na podatke

Da bismo stekli konkretnu predstavu o podacima, učitavamo skup i prikazujemo
prvih nekoliko redova. Detaljna analiza sledi u narednim sveskama — ovde samo
proveravamo kako podaci izgledaju.

In [1]:
# Uvozimo biblioteku pandas koja služi za rad sa tabelarnim podacima.
# Skraćenica "pd" je standardna konvencija — tako se pandas koristi svuda.
import pandas as pd

# Učitavamo CSV fajl u DataFrame (tabelu) i čuvamo ga u promenljivoj df.
# "../" znači da izlazimo iz foldera notebooks/ u glavni folder repoa,
# gde se nalazi CSV fajl.
df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Prikazujemo dimenzije skupa: df.shape vraća (broj_redova, broj_kolona).
# [0] uzima prvi element (redove), [1] drugi (kolone).
print("Broj redova:", df.shape[0])
print("Broj kolona:", df.shape[1])

# df.head() prikazuje prvih 5 redova tabele da vidimo kako podaci izgledaju.
# Kada je poslednja linija u ćeliji, Jupyter je automatski lepo prikaže.
df.head()

Broj redova: 7043
Broj kolona: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
